# Phase 1 — Shared corpus and tokenizer

Prepare the token streams used by all four models.

## Required files and runtime

- The official legal training corpus. Set `LOCAL_DATA` to the folder with the four `train_*.json` files, or enable Internet and an authorized `HF_TOKEN` Kaggle secret.
- A CPU session and enough disk space for the tokenizer, document index, and token streams.

## 1. Setup and corpus settings

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26,<3", "huggingface-hub>=0.30,<2",
    "ijson>=3.3,<4"
], check=True)


In [ ]:
import hashlib
import heapq
import json
import math
import os
import random
import re
import sqlite3
import time
from collections import Counter, defaultdict
from pathlib import Path
from types import SimpleNamespace

import numpy as np

In [ ]:
PREPARED_OUTPUT = Path("/kaggle/working/phase1_scratch_data")
LOCAL_DATA = None

CONFIG = {
    "seed": 42,
    "repo": "Exploration-Lab/CS-781-Capstone",
    "train_tokens": 810000000,
    "val_tokens": 2000000,
    "tokenizer_chars": 100000000,
    "vocab_size": 32768,
    "validation_fraction": 0.02,
    "mix": {
        "supreme_court": 0.35,
        "high_courts": 0.3,
        "acts": 0.2,
        "tribunals": 0.15
    },
    "model": {
        "width": 704,
        "layers": 16,
        "heads": 11,
        "context": 2048,
        "dropout": 0.0
    },
    "train": {
        "sequence_length": 2048,
        "micro_batch": 1,
        "accumulation": 16,
        "lr": 0.0003,
        "min_lr": 3e-05,
        "warmup_fraction": 0.03,
        "weight_decay": 0.1,
        "clip": 1.0,
        "eval_every": 250,
        "save_every": 100,
        "log_every": 10,
        "max_hours": 10.5
    }
}

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for name in ("HF_TOKEN", "HF"):
        try:
            token = secrets.get_secret(name)
            if token:
                os.environ["HF_TOKEN"] = token
                break
        except Exception:
            pass
except ImportError:
    pass

## 2. Corpus loading and split helpers

In [ ]:
SOURCES = ("supreme_court", "high_courts", "acts", "tribunals")
SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
PAD, UNK, BOS, EOS = range(4)
IMPLEMENTATION_VERSION = "single-notebook-handwritten-bpe-v2"


In [ ]:
def digest_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
def fingerprint(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


In [ ]:
def is_validation(text, cfg):
    h = hashlib.sha256((str(cfg['seed']) + '\x00' + text).encode('utf-8')).digest()
    return int.from_bytes(h[:8], 'big') / 2 ** 64 < cfg['validation_fraction']


In [ ]:
def quotas(total, mix):
    assert set(mix) == set(SOURCES) and abs(sum(mix.values()) - 1) < 1e-08
    result = {s: int(total * mix[s]) for s in SOURCES}
    result[SOURCES[-1]] += total - sum(result.values())
    return result


In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp, path)


In [ ]:
def read_config(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


In [ ]:
def stream_records(path):
    import ijson
    with open(path, 'rb') as f:
        first = f.read(1)
        while first and first.isspace():
            first = f.read(1)
        f.seek(0)
        if first == b'[':
            records = enumerate(ijson.items(f, 'item'))
        elif first == b'{':
            records = ijson.kvitems(f, '')
        else:
            raise ValueError(f'Expected JSON array/object: {path}')
        for key, record in records:
            if not isinstance(record, dict) or not isinstance(record.get('text'), str):
                raise ValueError(f'Record {key} has no string text field: {path}')
            if record['text'].strip():
                yield (str(key), record['text'])


In [ ]:
def locate(cfg, local):
    if local:
        paths = {s: Path(local) / f'train_{s}.json' for s in SOURCES}
    else:
        from huggingface_hub import hf_hub_download
        paths = {s: Path(hf_hub_download(cfg['repo'], f'Phase_1_Pretraining/Train/train_{s}.json', repo_type='dataset')) for s in SOURCES}
    for path in paths.values():
        if not path.is_file():
            raise FileNotFoundError(path)
    return paths


## 3. Train and verify the byte-level BPE tokenizer

In [ ]:
def split_bpe_pieces(text, max_chars=256):
    """Keep every character; separate words, punctuation and whitespace.

    An optional leading ASCII space can join a word or punctuation piece.
    Long pieces are split into bounded chunks to limit per-piece work.
    No normalization, strip(), split(), external tokenizer or regex package.
    """
    for match in re.finditer(' ?\\w+| ?[^\\w\\s]+|\\s+', text):
        piece = match.group(0)
        for start in range(0, len(piece), max_chars):
            yield piece[start:start + max_chars].encode('utf-8')


In [ ]:
def apply_bpe_merges(piece, merge_ranks):
    """Apply learned merges by rank, breaking occurrence ties left-to-right.

    Token IDs 0..3 are reserved. Byte b always has ID b+4. A linked list
    and a heap avoid repeatedly scanning every adjacent pair in a piece.
    """
    ids = [byte + 4 for byte in piece]
    size = len(ids)
    if size < 2 or not merge_ranks:
        return tuple(ids)
    previous = [i - 1 for i in range(size)]
    following = [i + 1 for i in range(size)]
    following[-1] = -1
    active = [True] * size
    queue = []

    def offer(left):
        if left < 0 or not active[left]:
            return
        right = following[left]
        if right < 0:
            return
        pair = (ids[left], ids[right])
        rank = merge_ranks.get(pair)
        if rank is not None:
            heapq.heappush(queue, (rank, left, right, pair[0], pair[1]))
    for left in range(size - 1):
        offer(left)
    while queue:
        rank, left, right, a, b = heapq.heappop(queue)
        if not active[left] or not active[right] or following[left] != right:
            continue
        if ids[left] != a or ids[right] != b:
            continue
        ids[left] = 260 + rank
        active[right] = False
        following[left] = following[right]
        if following[right] >= 0:
            previous[following[right]] = left
        offer(previous[left])
        offer(left)
    result = []
    position = 0
    while position >= 0:
        result.append(ids[position])
        position = following[position]
    return tuple(result)


In [ ]:
class ScratchByteBPE:
    """Our byte BPE vocabulary, encoder, decoder and JSON serialization.

    All base bytes and EVERY intermediate merge token are retained.
    Integer IDs identify symbols; concatenated decimal strings are never used.
    Literal '<s>' etc. in ordinary input are encoded as text, not control IDs.
    """
    FORMAT = 'handwritten-byte-bpe-v1'
    PATTERN = ' ?\\w+| ?[^\\w\\s]+|\\s+'

    def __init__(self, merges=None, max_piece_chars=256):
        self.specials = ['<pad>', '<unk>', '<s>', '</s>']
        self.max_piece_chars = max_piece_chars
        if not isinstance(max_piece_chars, int) or max_piece_chars < 1:
            raise ValueError('max_piece_chars must be a positive integer')
        self.merges = []
        self.merge_ranks = {}
        self.token_bytes = [None] * 4 + [bytes([b]) for b in range(256)]
        self._cache = {}
        for pair in merges or []:
            self.add_merge(tuple(pair))

    def add_merge(self, pair):
        if len(pair) != 2 or pair in self.merge_ranks:
            raise ValueError('Invalid or repeated merge pair')
        if any((type(token) is not int or not 4 <= token < len(self.token_bytes) for token in pair)):
            raise ValueError('A merge must reference earlier non-special token IDs')
        new_id = len(self.token_bytes)
        self.merge_ranks[pair] = len(self.merges)
        self.merges.append(pair)
        self.token_bytes.append(self.token_bytes[pair[0]] + self.token_bytes[pair[1]])
        self._cache.clear()
        return new_id

    def get_vocab_size(self):
        return len(self.token_bytes)

    def token_to_id(self, special):
        return self.specials.index(special) if special in self.specials else None

    def encode(self, text):
        if not isinstance(text, str):
            raise TypeError('encode expects a string')
        output = []
        for piece in split_bpe_pieces(text, self.max_piece_chars):
            encoded = self._cache.get(piece)
            if encoded is None:
                encoded = apply_bpe_merges(piece, self.merge_ranks)
                if len(self._cache) >= 65536:
                    self._cache.clear()
                self._cache[piece] = encoded
            output.extend(encoded)
        return SimpleNamespace(ids=output)

    def decode(self, ids, skip_special_tokens=True):
        chunks = []
        for value in ids:
            token = int(value)
            if not 0 <= token < len(self.token_bytes):
                raise ValueError(f'Unknown token ID: {token}')
            if token < 4:
                if not skip_special_tokens:
                    chunks.append(self.specials[token].encode('utf-8'))
            else:
                chunks.append(self.token_bytes[token])
        return b''.join(chunks).decode('utf-8', errors='replace')

    def save(self, path, training=None):
        payload = {'format': self.FORMAT, 'special_tokens': self.specials, 'byte_offset': 4, 'pattern': self.PATTERN, 'max_piece_chars': self.max_piece_chars, 'merges': self.merges, 'vocab_hex': [None if value is None else value.hex() for value in self.token_bytes]}
        if training is not None:
            payload['training'] = training
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix(path.suffix + '.tmp')
        temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding='utf-8')
        temporary.replace(path)

    @classmethod
    def from_file(cls, path):
        payload = json.loads(Path(path).read_text(encoding='utf-8'))
        if payload.get('format') != cls.FORMAT:
            raise ValueError('This is not our handwritten BPE format. Use a fresh data directory and retrain the tokenizer/token arrays.')
        if payload.get('special_tokens') != ['<pad>', '<unk>', '<s>', '</s>']:
            raise ValueError('Special-token IDs do not match')
        if payload.get('byte_offset') != 4 or payload.get('pattern') != cls.PATTERN:
            raise ValueError('Byte mapping or splitting rules do not match')
        tokenizer = cls(payload['merges'], payload['max_piece_chars'])
        expected = [None if value is None else value.hex() for value in tokenizer.token_bytes]
        if payload.get('vocab_hex') != expected:
            raise ValueError('Vocabulary is inconsistent with the merge history')
        return tokenizer


In [ ]:
def build_bpe_statistics(piece_counts, tokenizer):
    """Create weighted pair counts and exact pair-to-piece memberships."""
    words, weights = ([], [])
    frequencies = Counter()
    members = defaultdict(set)
    for piece, count in piece_counts.items():
        symbols = list(apply_bpe_merges(piece, tokenizer.merge_ranks))
        word_id = len(words)
        words.append(symbols)
        weights.append(count)
        for pair, occurrences in Counter(zip(symbols, symbols[1:])).items():
            frequencies[pair] += occurrences * count
            members[pair].add(word_id)
    return (words, weights, frequencies, members)


In [ ]:
def learn_bpe_merges(tokenizer, texts, vocab_size, min_frequency=2, checkpoint_path=None, training_signature=None, save_every=1000, stop_after=None):
    """Handwritten weighted BPE training with incremental counts and a heap.

    Select the most frequent pair; ties use integer pair order. Merge every
    non-overlapping occurrence left-to-right in affected pieces. Update only
    those pieces. Stale heap entries are discarded and periodically compacted.
    Existing merges are replayed when resuming a tokenizer checkpoint.
    """
    if not 260 <= vocab_size <= 65536:
        raise ValueError('Vocabulary must fit the uint16 token files (260..65536)')
    if tokenizer.get_vocab_size() > vocab_size:
        raise ValueError('Existing vocabulary exceeds requested size')
    piece_counts = Counter()
    for text in texts:
        piece_counts.update(split_bpe_pieces(text, tokenizer.max_piece_chars))
    if not piece_counts:
        raise ValueError('No text available to train BPE')
    print(f'Unique text pieces: {len(piece_counts):,}', flush=True)
    words, weights, frequencies, members = build_bpe_statistics(piece_counts, tokenizer)
    del piece_counts
    queue = [(-count, a, b) for (a, b), count in frequencies.items()]
    heapq.heapify(queue)
    started, learned = (time.monotonic(), 0)
    metadata = {'signature': training_signature, 'vocab_size': vocab_size, 'min_frequency': min_frequency}
    while tokenizer.get_vocab_size() < vocab_size and queue:
        negative_count, a, b = heapq.heappop(queue)
        pair = (a, b)
        if frequencies.get(pair, 0) != -negative_count:
            continue
        if -negative_count < min_frequency:
            break
        affected = list(members[pair])
        new_id = tokenizer.add_merge(pair)
        changed = set()
        for word_id in affected:
            old = words[word_id]
            old_pairs = Counter(zip(old, old[1:]))
            merged, position = ([], 0)
            while position < len(old):
                if position + 1 < len(old) and (old[position], old[position + 1]) == pair:
                    merged.append(new_id)
                    position += 2
                else:
                    merged.append(old[position])
                    position += 1
            new_pairs = Counter(zip(merged, merged[1:]))
            for current in old_pairs.keys() | new_pairs.keys():
                delta = (new_pairs[current] - old_pairs[current]) * weights[word_id]
                if delta:
                    frequencies[current] += delta
                    if frequencies[current] < 0:
                        raise AssertionError('Negative pair frequency')
                    changed.add(current)
                if current not in new_pairs:
                    members[current].discard(word_id)
                    if not members[current]:
                        del members[current]
                elif current not in old_pairs:
                    members[current].add(word_id)
            words[word_id] = merged
        for current in changed:
            count = frequencies[current]
            if count:
                heapq.heappush(queue, (-count, current[0], current[1]))
            else:
                del frequencies[current]
        learned += 1
        if len(queue) > 4 * len(frequencies) + 10000:
            queue = [(-count, x, y) for (x, y), count in frequencies.items()]
            heapq.heapify(queue)
        if learned % save_every == 0:
            print(f'Vocabulary: {tokenizer.get_vocab_size():,}/{vocab_size:,}; elapsed: {(time.monotonic() - started) / 60:.1f} min', flush=True)
            if checkpoint_path is not None:
                tokenizer.save(checkpoint_path, training=metadata)
        if stop_after is not None and learned >= stop_after:
            break
    if checkpoint_path is not None:
        tokenizer.save(checkpoint_path, training=metadata)
    tokenizer._cache.clear()
    print(f'BPE vocabulary: {tokenizer.get_vocab_size():,}', flush=True)
    return tokenizer


In [ ]:
def train_tokenizer(cfg, paths, out):
    sample_dir = out / 'tokenizer_samples'
    sample_dir.mkdir(exist_ok=True)
    sizes = quotas(cfg['tokenizer_chars'], cfg['mix'])
    sample_paths = []
    for source in SOURCES:
        rng = random.Random(f"{cfg['seed']}:{source}")
        chunk = min(8192, sizes[source])
        if chunk < 1:
            raise ValueError('Tokenizer sample budget too small')
        capacity = max(1, sizes[source] // chunk)
        reservoir, eligible = ([], 0)
        for _, text in stream_records(paths[source]):
            if is_validation(text, cfg):
                continue
            eligible += 1
            j = eligible - 1 if eligible <= capacity else rng.randrange(eligible)
            if j < capacity:
                start = rng.randrange(max(1, len(text) - chunk + 1))
                excerpt = text[start:start + chunk]
                if j == len(reservoir):
                    reservoir.append(excerpt)
                else:
                    reservoir[j] = excerpt
            if eligible % 100000 == 0:
                print(source, 'eligible documents scanned:', eligible, flush=True)
        if not reservoir:
            raise ValueError(f'No tokenizer training text: {source}')
        sample_path = sample_dir / f'{source}.jsonl'
        with sample_path.open('w', encoding='utf-8') as f:
            for text in reservoir:
                f.write(json.dumps(text, ensure_ascii=False) + '\n')
        print(source, 'sample characters:', sum(map(len, reservoir)), flush=True)
        sample_paths.append(sample_path)
        del reservoir
    signature = {path.name: digest_file(path) for path in sample_paths}
    progress = out / 'tokenizer_progress.json'
    metadata = {'signature': signature, 'vocab_size': cfg['vocab_size'], 'min_frequency': 2}
    if progress.exists():
        saved = read_config(progress)
        if saved.get('training') != metadata:
            raise ValueError('Tokenizer progress belongs to different samples/settings; use a fresh output directory')
        tokenizer = ScratchByteBPE.from_file(progress)
        print('Resuming BPE at vocabulary size', tokenizer.get_vocab_size(), flush=True)
    else:
        tokenizer = ScratchByteBPE()

    def samples():
        for path in sample_paths:
            with path.open(encoding='utf-8') as file:
                for line in file:
                    yield json.loads(line)
    learn_bpe_merges(tokenizer, samples(), vocab_size=cfg['vocab_size'], min_frequency=2, checkpoint_path=progress, training_signature=signature)
    if tokenizer.get_vocab_size() != cfg['vocab_size']:
        raise ValueError(f'Only learned {tokenizer.get_vocab_size()} entries. Increase the sample budget; no fake vocabulary entries are added.')
    tokenizer.save(out / 'tokenizer.json')
    return tokenizer


In [ ]:
def validate_tokenizer(tok):
    assert [tok.token_to_id(s) for s in SPECIALS] == list(range(4))
    for text in ['Section 498-A: ₹5,000.\n\tभारत — न्यायालय', 'a  b\r\nc', '😀 é é', '123/2026 (SC)']:
        ids = tok.encode(text).ids
        assert UNK not in ids and tok.decode(ids, skip_special_tokens=False) == text


In [ ]:
def verify_scratch_tokenizer(tokenizer):
    validate_tokenizer(tokenizer)
    probes = ["", " ", "a  b\r\n\tc", "😀 café e\u0301 भारत ₹", "<pad> <s> literal text", "x" * 1000]
    for text in probes:
        ids = tokenizer.encode(text).ids
        assert all(4 <= value < tokenizer.get_vocab_size() for value in ids)
        assert tokenizer.decode(ids, skip_special_tokens=False) == text
    assert tokenizer.token_bytes[4:260] == [bytes([value]) for value in range(256)]
    print("Handwritten tokenizer: byte coverage, whitespace, Unicode and literal special-token text checks passed")


## 4. Build the shared dataset

In [ ]:
def prepare(cfg, paths, out, tok):
    """Honor source quotas where possible, then fill shortages without repeats.

    The requested total token budgets remain fixed. High Courts, Tribunals,
    Acts, then Supreme Court supply any deficit left after the first pass.
    Exact-document deduplication and the content-based held-out split apply
    throughout. A partially used document can supply its unused tail once.
    """
    import contextlib
    out = Path(out)
    if (out / 'manifest.json').exists():
        raise ValueError('This dataset is already complete. Use prepare_shared_data to verify/reuse it.')
    totals = {'train': cfg['train_tokens'], 'val': cfg['val_tokens']}
    requested = {split: quotas(total, cfg['mix']) for split, total in totals.items()}
    counts = {source: {'train': 0, 'val': 0} for source in SOURCES}
    examples, example_counts = ({}, {source: 0 for source in SOURCES})
    fallback_order = ('high_courts', 'tribunals', 'acts', 'supreme_court')
    with contextlib.ExitStack() as stack:
        db = sqlite3.connect(out / 'seen_capacity.sqlite')
        stack.callback(db.close)
        db.execute('DROP TABLE IF EXISTS seen')
        db.execute('CREATE TABLE seen (hash TEXT PRIMARY KEY, source TEXT, original_id TEXT, used INTEGER, complete INTEGER)')
        index = stack.enter_context((out / 'documents.jsonl').open('w', encoding='utf-8'))
        handles = {}
        for source in SOURCES:
            for split in totals:
                file = stack.enter_context((out / f'{source}.{split}.bin').open('wb'))
                np.array([BOS], dtype='<u2').tofile(file)
                handles[source, split] = file

        def ingest(source, limits):
            added_documents = 0
            for original_id, text in stream_records(paths[source]):
                if all((counts[source][split] >= limits[split] for split in totals)):
                    break
                split = 'val' if is_validation(text, cfg) else 'train'
                remaining = limits[split] - counts[source][split]
                if remaining <= 0:
                    continue
                original_id = str(original_id)
                h = fingerprint(text)
                old = db.execute('SELECT source, original_id, used, complete FROM seen WHERE hash=?', (h,)).fetchone()
                if old is not None:
                    owner_source, owner_id, used, complete = old
                    if complete or owner_source != source or owner_id != original_id:
                        continue
                else:
                    used = 0
                ids = tok.encode(text).ids + [EOS]
                keep = ids[used:used + remaining]
                if not keep:
                    raise AssertionError('Invalid document offset in deduplication state')
                start = counts[source][split] + 1
                np.asarray(keep, dtype='<u2').tofile(handles[source, split])
                end_used = used + len(keep)
                complete = end_used == len(ids)
                index.write(json.dumps({'source': source, 'split': split, 'original_id': original_id, 'sha256': h, 'start': start, 'end': start + len(keep), 'document_token_start': used, 'document_token_end': end_used, 'truncated': not complete}) + '\n')
                db.execute('INSERT OR REPLACE INTO seen VALUES (?, ?, ?, ?, ?)', (h, source, original_id, end_used, int(complete)))
                counts[source][split] += len(keep)
                if split == 'val' and example_counts[source] < 64:
                    key = f'{source}_{h[:20]}'
                    words = text.split()
                    if key not in examples and len(words) >= 640:
                        examples[key] = {'prompt': ' '.join(words[:512]), 'reference': ' '.join(words[512:640])}
                        example_counts[source] += 1
                added_documents += 1
                if added_documents % 1000 == 0:
                    db.commit()
                    print(source, counts[source], flush=True)
            db.commit()
            print(source, 'collected:', counts[source], flush=True)
        for source in SOURCES:
            ingest(source, {split: requested[split][source] for split in totals})
        first_pass = {source: dict(value) for source, value in counts.items()}
        for source in fallback_order:
            missing = {split: total - sum((value[split] for value in counts.values())) for split, total in totals.items()}
            if not any(missing.values()):
                break
            print('Filling remaining token budget:', missing, 'from', source, flush=True)
            ingest(source, {split: counts[source][split] + missing[split] for split in totals})
        missing = {split: total - sum((value[split] for value in counts.values())) for split, total in totals.items()}
        if any(missing.values()):
            raise ValueError(f'All four sources have been checked, but the unique-data budget is still short: {missing}. The tokenizer is saved. Reduce the total budget explicitly; documents were not repeated.')
    if not examples:
        raise ValueError('No held-out documents with 640 words; increase validation budget')
    actual_mix = {split: {source: counts[source][split] / total for source in SOURCES} for split, total in totals.items()}
    report = {'policy': 'requested quotas first; deficits filled by unused data', 'fallback_order': list(fallback_order), 'requested_mix': cfg['mix'], 'requested_targets': requested, 'first_pass_targets': first_pass, 'actual_targets': counts, 'actual_mix': actual_mix, 'totals': totals}
    manifest = {'config': cfg, 'tokenizer_sha256': digest_file(out / 'tokenizer.json'), 'files': {}, 'document_index': 'documents.jsonl', 'format': 'little-endian uint16', 'boundary': 'EOS between documents; one BOS at beginning of each source/split', 'allocation': report}
    for source in SOURCES:
        for split in totals:
            name = f'{source}.{split}.bin'
            assert (out / name).stat().st_size == 2 * (counts[source][split] + 1)
            manifest['files'][name] = {'targets': counts[source][split], 'bytes': (out / name).stat().st_size, 'sha256': digest_file(out / name)}
    write_json(out / 'dev_prompts.json', {key: {'text': value['prompt']} for key, value in examples.items()})
    write_json(out / 'dev_truth.json', {key: {'text': value['reference']} for key, value in examples.items()})
    write_json(out / 'data_mix_report.json', report)
    write_json(out / 'manifest.json', manifest)
    print('Prepared training targets:', totals['train'], 'validation targets:', totals['val'])
    print('Actual training mix:', {source: f'{share:.2%}' for source, share in actual_mix['train'].items()})


In [ ]:
def prepare_shared_data(cfg, out, local_data=None):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    manifest_path = out / "manifest.json"
    if manifest_path.exists():
        manifest = read_config(manifest_path)
        verify_scratch_tokenizer(ScratchByteBPE.from_file(out / "tokenizer.json"))
        if manifest["config"] != cfg:
            raise ValueError("Existing prepared data uses different settings; use a new directory")
        assert digest_file(out / "tokenizer.json") == manifest["tokenizer_sha256"]
        for name, meta in manifest["files"].items():
            assert digest_file(out / name) == meta["sha256"], name
        print("Verified existing prepared dataset:", out)
        return

    paths = locate(cfg, local_data)
    provenance = {
        "config": cfg,
        "inputs": {source: {"bytes": path.stat().st_size, "sha256": digest_file(path)}
                   for source, path in paths.items()}
    }
    stamp = out / "tokenizer_provenance.json"
    if (out / "tokenizer.json").exists():
        if not stamp.exists() or read_config(stamp) != provenance:
            raise ValueError("Tokenizer provenance mismatch; choose a fresh output directory")
        tokenizer = ScratchByteBPE.from_file(out / "tokenizer.json")
    else:
        tokenizer = train_tokenizer(cfg, paths, out)
        write_json(stamp, provenance)
    verify_scratch_tokenizer(tokenizer)
    verify_scratch_tokenizer(ScratchByteBPE.from_file(out / "tokenizer.json"))
    prepare(cfg, paths, out, tokenizer)


In [ ]:
prepare_shared_data(CONFIG, PREPARED_OUTPUT, LOCAL_DATA)
manifest = read_config(PREPARED_OUTPUT / "manifest.json")
training_targets = sum(info["targets"] for name, info in manifest["files"].items()
                       if ".train." in name)
print(f"Training targets: {training_targets:,}")
print("Prepared dataset:", PREPARED_OUTPUT)

Prepared training targets: 810000000 validation targets: 2000000
Actual training mix: {'supreme_court': '31.11%', 'high_courts': '34.92%', 'acts': '18.97%', 'tribunals': '15.00%'}
Training targets: 810,000,000
Prepared dataset: /kaggle/working/phase1_scratch_data
